# Диагностика Jupyter kernel

Выберите kernel проекта, затем выполните **Restart Kernel and Run All Cells**. Отчёт должен показывать Python и prefix ожидаемого окружения.

In [ ]:
import importlib.metadata
import json
import os
import platform
import sys
from pathlib import Path

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

diagnostic = {
    "executable": str(Path(sys.executable).resolve()),
    "prefix": str(Path(sys.prefix).resolve()),
    "base_prefix": str(Path(sys.base_prefix).resolve()),
    "python_version": platform.python_version(),
    "ipykernel_version": package_version("ipykernel"),
    "cwd": str(Path.cwd().resolve()),
    "virtual_environment": os.environ.get("VIRTUAL_ENV"),
}
print(json.dumps(diagnostic, ensure_ascii=False, indent=2))

## Проверка ожидаемого окружения

Укажите корень `.venv`, если хотите превратить диагностику в строгую проверку. Пустое значение оставляет только проверку наличия `ipykernel`.

In [ ]:
expected_prefix = ""  # Например: /absolute/path/to/project/.venv

assert diagnostic["ipykernel_version"], "ipykernel отсутствует в kernel environment"
if expected_prefix:
    actual = Path(diagnostic["prefix"]).resolve()
    expected = Path(expected_prefix).expanduser().resolve()
    assert actual == expected, f"Kernel использует {actual}, ожидалось {expected}"
print("Kernel runtime подтверждён")

## Сверка с kernelspec

В терминале выполните `jupyter kernelspec list --json`, найдите каталог выбранного kernel и проверьте его командой `python3 outputs/kernel_diagnostic.py check /path/to/kernel.json`. Поле `argv[0]` должно вести к тому же Python, который показан в `sys.executable`.